In [2]:
import numpy as np

In [3]:
eps0 = 8.854187817e-12
epsr = 80.1
epse = eps0 * epsr
T = 298
mu = 1.0e-3
rho = 1.0e3
L = 1e-5
kb = 1.38064852e-23
ez = 1.60217662e-19
mol = 6.022e23

In [4]:
alpha = 0.01
zeta0 = alpha *kb * T / ez
print("zeta_0 = ", -zeta0 * 1000.0, "mV")
E = 1.0e5
E_ = E * L / zeta0
print("E_x = ", E_)

U = epse * zeta0 * E / mu
print("U_sh = ", U, "m/s")

Re = rho * U * L / mu
print("Re = ", Re)

D = 4.0e-9
Pe = U * L / D
print("Pe = ", Pe)

kappa = 128.0
delta = kappa * kappa / (2.0 * alpha)
print("delta = ", delta)

n0 = delta * epse * zeta0 / (ez * L * L)
print("n_0 = ", n0/mol * 1.0e3, "mol/L = ", n0, "m^-3")
print("Sc =  ", Pe/Re)
print("dt = ", L / U)

zeta_0 =  -0.25679644417729675 mV
E_x =  3894.1349176532312
U_sh =  1.821252881934316e-05 m/s
Re =  0.00018212528819343163
Pe =  0.04553132204835791
delta =  819200.0
n_0 =  15.463544715432247 mol/L =  9.312146627633299e+21 m^-3
Sc =   250.0
dt =  0.5490725697234973


In [15]:
exp = np.exp
tanh = np.tanh
pow = np.power
log = np.log
psi_up = -1.0
psi_down = -1.0
N = 320
dx = 1.0/N
dxb = 0.5*dx
tol = 1e-12
alpha = 1.0
kappa = 20.0

def eval_prop_out(y, fun, order=1):
    if(order==1):
        if y>1.0+tol:
            last = 1.0-dxb
            dif = y-last
            last_val = fun(last)
            bd_val = fun(1.0)
        else :
            last = -1.0+dxb
            dif = y-last
            last_val = fun(last)
            bd_val = fun(-1.0)
        slope = (bd_val-last_val)/dxb
        val = last_val + dif*slope
    elif(order==2):
        if y>1.0+tol:
            last = 1.0-dxb
            last2 = last-dx
            difb = y-1.0
            dif1 = y-last
            dif2 = y-last2
            last_val = fun(last)
            last2_val = fun(last2)
            bd_val = fun(1.0)
        else :
            last = -1.0+dxb
            last2 = last+dx
            difb = y+1.0
            dif1 = y-last
            dif2 = y-last2
            last_val = fun(last)
            last2_val = fun(last2)
            bd_val = fun(-1.0)
        w_bd = dif1*dif2/(dxb*(dx+dxb))
        w_last = difb*dif2/(-dxb*dx)
        w_last2 = difb*dif1/(-(dx+dxb)*-dx)
        val = w_bd*bd_val + w_last*last_val + w_last2*last2_val

    return val

def quot_m1(y):
    ek1my = exp(kappa*(1.0-y))
    tanhma4 = tanh(-alpha/4.0)
    quot = (ek1my+tanhma4)/(ek1my-tanhma4)
    return quot
def quot_p1(y):
    ek1py = exp(kappa*(1.0+y))
    tanhma4 = tanh(-alpha/4.0)
    quot = (ek1py+tanhma4)/(ek1py-tanhma4)
    return quot

def psi(y):
    infsol_m1 = -2.0/alpha*log(quot_m1(y))
    infsol_p1 = -2.0/alpha*log(quot_p1(y))
    return psi_up*infsol_m1 + psi_down*infsol_p1

def dpsidx(y):
    ek1my = exp(kappa*(1.0-y))
    ek1py = exp(kappa*(1.0+y))
    tanhma4 = tanh(-alpha/4.0)
    infsol_m1 = 4.0/alpha*tanhma4*(-kappa*ek1my)/(ek1my*ek1my-tanhma4*tanhma4)
    infsol_p1 = 4.0/alpha*tanhma4*(kappa*ek1py)/(ek1py*ek1py-tanhma4*tanhma4)
    return psi_up*infsol_m1 + psi_down*infsol_p1


def nplus(y):
    np_pb = pow(quot_m1(y), 2*psi_up) * pow(quot_p1(y), 2*psi_down)
    return np_pb

def d2psidx2(y):
    delta = kappa*kappa/(2.0*alpha)
    npy = nplus(y)
    return -delta*(npy - 1.0/npy)

def dnpdx(y):
    return -alpha*dpsidx(y)*nplus(y)

def d2npdx2(y):
    dpsi = dpsidx(y)
    return (-d2psidx2(y) + alpha*dpsi*dpsi)*alpha*nplus(y)

order = 1

def np_num(y):
    if (y<1.0+tol and y>-1.0-tol):
        return nplus(y)
    return eval_prop_out(y, nplus,order)

def psi_num(y):
    if (y<1.0+tol and y>-1.0-tol):
        return psi(y)
    return eval_prop_out(y, psi,order)

pc = 1.0 - dxb
pl = pc  - dx
pr = pc  + dx
nl = np_num(pl)
nr = np_num(pr)
nc = np_num(pc)
psil = psi_num(pl)
psir = psi_num(pr)
psic = psi_num(pc)

dpsidxl_num = (psir - psic)/(dx)
dpsidxr_num = (psic - psil)/(dx)
vrp = max(dpsidxr_num,0.0)
vrm = min(dpsidxr_num,0.0)
vlp = max(dpsidxl_num,0.0)
vlm = min(dpsidxl_num,0.0)

Fr = vrp*nc + vrm*nr
Fl = vlp*nl + vlm*nc
d_ndpsidx_dx_num = (Fr - Fl)/(dx)
#dpsidx_num = 1/(12*dx)*(psi_num(pc-2*dx)-8*psi_num(pc-dx)+8*psi_num(pc+dx)-psi_num(pc+2*dx))
d2npdx2_num = (nr-2*nc+nl)/(dx*dx)
#dndx_num = 1/(12*dx)*(np_num(pc-2*dx)-8*np_num(pc-dx)+8*np_num(pc+dx)-np_num(pc+2*dx))
print(1/(3*dxb*dxb)*(2*nplus(1.0)-3*nc + nl)- d2npdx2(pc))
print(d2npdx2_num - d2npdx2(pc))
print(1/(420*dxb*dxb)*(352*nplus(1.0)-560*nc + 245*nl - 42*nplus(pc-2*dx) + 5*nplus(pc-3*dx))- d2npdx2(pc))


print(-alpha*(dnpdx(pc)*dpsidx(pc)+nplus(pc)*d2psidx2(pc)))
print(psi_num(pr) - psi(pr))
print((d2npdx2_num, -alpha*d_ndpsidx_dx_num))


N = 256
dx = 1.0/N
pc = 1.0 - 0.5*dx
psil = psi(pc-dx)
psill = psi(pc-2*dx)
psilll = psi(pc-3*dx)
psir = psi(pc+dx)
psirr = psi(pc+2*dx)
psirrr = psi(pc+3*dx)
psic = psi(pc)

print("d2psi_err = ",(2*psilll -27*psill + 270*psil - 490*psic + 270*psir -27*psirr + 2*psirrr)/(180*dx*dx) - d2psidx2(pc))

    

-60.019942359257584
-608.1217506108098
-0.09832441922799262
2252.4271753640114
0.0011479030574226279
(1644.3054247532016, 187.478527225403)
d2psi_err =  0.004357321514021351


-0.24317330964795714
